In [3]:
import pandas as pd

df = pd.read_csv("twcs.csv")

# load the dataset

In [4]:
print(df.head(5))

#chek the type of the data

   tweet_id   author_id  inbound                      created_at  \
0         1  sprintcare    False  Tue Oct 31 22:10:47 +0000 2017   
1         2      115712     True  Tue Oct 31 22:11:45 +0000 2017   
2         3      115712     True  Tue Oct 31 22:08:27 +0000 2017   
3         4  sprintcare    False  Tue Oct 31 21:54:49 +0000 2017   
4         5      115712     True  Tue Oct 31 21:49:35 +0000 2017   

                                                text response_tweet_id  \
0  @115712 I understand. I would like to assist y...                 2   
1      @sprintcare and how do you propose we do that               NaN   
2  @sprintcare I have sent several private messag...                 1   
3  @115712 Please send us a Private Message so th...                 3   
4                                 @sprintcare I did.                 4   

   in_response_to_tweet_id  
0                      3.0  
1                      1.0  
2                      4.0  
3                      5.0  
4

In [5]:
df["author_id"] = df["author_id"].astype(str)

companies = df[
    ~df["author_id"].str.match(r"^\d+$")
]["author_id"].unique()

print("Number of companies:", len(companies))
print(companies)

#explore the dataset to list the distinct companies 

Number of companies: 108
<StringArray>
[     'sprintcare',    'Ask_Spectrum',  'VerizonSupport',  'ChipotleTweets',
  'AskPlayStation', 'marksandspencer',  'MicrosoftHelps',      'ATVIAssist',
       'AdobeCare',      'AmazonHelp',
 ...
 'mediatemplehelp',       'AskTigogh',  'PandoraSupport',         'askvisa',
      'OPPOCareIN', 'ask_progressive',  'PearsonSupport',         'CarlsJr',
  'HotelTonightCX',    'KeyBank_Help']
Length: 108, dtype: str


In [6]:
company_counts = (
    df[~df["author_id"].str.match(r"^\d+$")]
    ["author_id"]
    .value_counts()
)

print(company_counts)

# check which company has the highest amount of tweets 

author_id
AmazonHelp        169840
AppleSupport      106860
Uber_Support       56270
SpotifyCares       43265
Delta              42253
                   ...  
JackBox              266
OfficeSupport        218
AskDSC               210
CarlsJr              196
HotelTonightCX       152
Name: count, Length: 108, dtype: int64


In [7]:
amazon_df = df[
    (df["author_id"] == "AmazonHelp") |
    (df["text"].str.contains("@AmazonHelp", case=False, na=False))
]

# filtering the dataset 

In [8]:
print(amazon_df.head())
print("Number of tweets:", len(amazon_df))

     tweet_id   author_id  inbound                      created_at  \
181       269  AmazonHelp    False  Wed Nov 22 09:23:01 +0000 2017   
182       270      115770     True  Wed Nov 22 09:24:30 +0000 2017   
183       271      115770     True  Wed Nov 22 09:30:36 +0000 2017   
184       273  AmazonHelp    False  Wed Nov 22 09:40:27 +0000 2017   
185       274      115770     True  Wed Nov 22 09:44:04 +0000 2017   

                                                  text response_tweet_id  \
181  @115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...           270,271   
182      @AmazonHelp ありがとうございます。\n今、電話で主人が対応していただいてます。               NaN   
183  @AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎ...               273   
184  @115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...               274   
185                      @AmazonHelp こちらこそありがとうございました。               275   

     in_response_to_tweet_id  
181                    272.0  
182                    269.0  
183                    269.0 

In [9]:
# understanding the dataset

print("Rows:", amazon_df.shape[0])
print("Columns:", amazon_df.shape[1])

Rows: 305008
Columns: 7


In [10]:
missing = pd.DataFrame({
    "Missing Count": amazon_df.isnull().sum(),
    "Missing %": amazon_df.isnull().mean() * 100
})

print(missing)

                         Missing Count  Missing %
tweet_id                             0   0.000000
author_id                            0   0.000000
inbound                              0   0.000000
created_at                           0   0.000000
text                                 0   0.000000
response_tweet_id               119543  39.193398
in_response_to_tweet_id          22907   7.510295


In [ ]:
# standardizing the date and time

amazon_df["created_at"] = pd.to_datetime(
    amazon_df["created_at"]
)

/tmp/ipykernel_24961/2736087902.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  amazon_df["created_at"] = pd.to_datetime(


In [15]:
amazon_df["is_reply"] = amazon_df["in_response_to_tweet_id"].notna()

amazon_df["has_response"] = amazon_df["response_tweet_id"].notna()

In [19]:
amazon_df["clean_text"] = (
    amazon_df["text"]
    .str.replace(r"http\S+|www\S+", "", regex=True)
    .str.replace(r"@\w+", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [42]:
empty_clean = amazon_df[
    amazon_df["clean_text"].fillna("").str.strip() == ""
]

print("Empty after cleaning:", len(empty_clean))

print(
    empty_clean[
        ["tweet_id", "author_id", "inbound", "text", "clean_text"]
    ].head()
)

Empty after cleaning: 1972
       tweet_id author_id  inbound  \
7552      11076    118089     True   
9801      13565    118829     True   
10131     13917    118925     True   
12751     16877    119703     True   
14419     18729    120177     True   

                                               text clean_text  
7552    @118088 @AmazonHelp https://t.co/JeO1QDroxC             
9801           @AmazonHelp https://t.co/saX0kAZVCW!             
10131          @AmazonHelp  https://t.co/DJzdya8ZMM             
12751  @AmazonHelp @119704  https://t.co/rUjHdeMm9G             
14419          @AmazonHelp  https://t.co/LzRfyE5m3q             


In [43]:
print(
    "Empty original text:",
    amazon_df["text"].fillna("").str.strip().eq("").sum()
)

Empty original text: 0


In [45]:
customer_df = amazon_df[
    amazon_df["inbound"] == True
].copy()

amazon_response_df = amazon_df[
    (amazon_df["author_id"] == "AmazonHelp") &
    (amazon_df["inbound"] == False)
].copy()

print("Customer tweets:", len(customer_df))
print("Amazon responses:", len(amazon_response_df))

Customer tweets: 135160
Amazon responses: 169840


In [47]:
customer_df = customer_df.rename(
    columns={
        "tweet_id": "customer_tweet_id",
        "author_id": "customer_id",
        "created_at": "customer_time",
        "text": "customer_query",
        "clean_text": "clean_customer_query"
    }
)

amazon_response_df = amazon_response_df.rename(
    columns={
        "tweet_id": "amazon_tweet_id",
        "in_response_to_tweet_id": "customer_tweet_id",
        "created_at": "amazon_time",
        "text": "amazon_response",
        "clean_text": "clean_amazon_response"
    }
)

In [48]:
conversation_df = amazon_response_df.merge(
    customer_df,
    on="customer_tweet_id",
    how="inner"
)

In [49]:
print(
    conversation_df[
        [
            "customer_tweet_id",
            "customer_query",
            "amazon_tweet_id",
            "amazon_response"
        ]
    ].head(10)
)

   customer_tweet_id                                     customer_query  \
0              271.0  @AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎ...   
1              274.0                      @AmazonHelp こちらこそありがとうございました。   
2              616.0  @AmazonHelp 3 different people have given 3 di...   
3              623.0               @AmazonHelp Okay, danke für die Info   
4              627.0  @AmazonHelp @115826 Yeah this is crazy we’re l...   
5              634.0  @115821 @AmazonHelp why is my order at my loca...   
6              638.0                 @AmazonHelp Hi ready for some help   
7              640.0  @AmazonHelp Is the Echo Show no longer supported?   
8              643.0  Bought an @115821 Echo Show and it won’t recog...   
9              646.0  .@AmazonHelp Item has not been delivered but t...   

   amazon_tweet_id                                    amazon_response  
0              273  @115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...  
1              275  @115770 恐れ

In [50]:
print("Customer tweets:", len(customer_df))
print("Amazon responses:", len(amazon_response_df))
print("Matched pairs:", len(conversation_df))

Customer tweets: 135160
Amazon responses: 169840
Matched pairs: 100015


In [51]:
empty_clean = amazon_df[
    amazon_df["clean_text"].fillna("").str.strip() == ""
].copy()

print("Empty after cleaning:", len(empty_clean))

print(
    empty_clean[
        [
            "tweet_id",
            "author_id",
            "inbound",
            "text",
            "clean_text"
        ]
    ].head(30).to_string(index=False)
)

Empty after cleaning: 1972
 tweet_id author_id  inbound                                         text clean_text
    11076    118089     True  @118088 @AmazonHelp https://t.co/JeO1QDroxC           
    13565    118829     True         @AmazonHelp https://t.co/saX0kAZVCW!           
    13917    118925     True         @AmazonHelp  https://t.co/DJzdya8ZMM           
    16877    119703     True @AmazonHelp @119704  https://t.co/rUjHdeMm9G           
    18729    120177     True         @AmazonHelp  https://t.co/LzRfyE5m3q           
    18919    120227     True         @AmazonHelp  https://t.co/9QpAMYZbaJ           
    19109    120259     True                                  @AmazonHelp           
    21563    120701     True         @AmazonHelp  https://t.co/RhLDADmDNX           
    22498    120917     True          @AmazonHelp https://t.co/SGdJV5T2f4           
    22502    120918     True          @AmazonHelp https://t.co/QCt7kMtESE           
    22563    120930     True         @

In [52]:
amazon_clean_df = amazon_df[
    amazon_df["clean_text"].fillna("").str.strip() != ""
].copy()

print("Original Amazon-related tweets:", len(amazon_df))
print("After removing empty cleaned tweets:", len(amazon_clean_df))
print("Removed:", len(amazon_df) - len(amazon_clean_df))

Original Amazon-related tweets: 305008
After removing empty cleaned tweets: 303036
Removed: 1972


In [53]:
customer_df = amazon_clean_df[
    amazon_clean_df["inbound"] == True
].copy()

amazon_response_df = amazon_clean_df[
    (amazon_clean_df["author_id"] == "AmazonHelp") &
    (amazon_clean_df["inbound"] == False)
].copy()

print("Customer tweets:", len(customer_df))
print("Amazon responses:", len(amazon_response_df))

Customer tweets: 133204
Amazon responses: 169826


In [54]:
customer_df = customer_df.rename(columns={
    "tweet_id": "customer_tweet_id",
    "author_id": "customer_id",
    "created_at": "customer_time",
    "text": "customer_query",
    "clean_text": "clean_customer_query"
})

amazon_response_df = amazon_response_df.rename(columns={
    "tweet_id": "amazon_tweet_id",
    "in_response_to_tweet_id": "customer_tweet_id",
    "created_at": "amazon_time",
    "text": "amazon_response",
    "clean_text": "clean_amazon_response"
})

conversation_df = amazon_response_df.merge(
    customer_df,
    on="customer_tweet_id",
    how="inner"
)

print("Matched pairs:", len(conversation_df))

Matched pairs: 98828


In [55]:
print(
    "Duplicate customer tweets:",
    conversation_df["customer_tweet_id"].duplicated().sum()
)

print(
    "Duplicate Amazon responses:",
    conversation_df["amazon_tweet_id"].duplicated().sum()
)

Duplicate customer tweets: 7040
Duplicate Amazon responses: 0


In [56]:
print(
    conversation_df[
        [
            "customer_tweet_id",
            "customer_id",
            "customer_query",
            "amazon_tweet_id",
            "amazon_response"
        ]
    ].head(10).to_string(index=False)
)

 customer_tweet_id customer_id                                                                                                                                                     customer_query  amazon_tweet_id                                                                                                                      amazon_response
             271.0      115770                                                                                                   @AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎてるので買い直しになるんでしょうね。              273                                                                @115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました。リプライいただきありがとうございました。ET
             274.0      115770                                                                                                                                      @AmazonHelp こちらこそありがとうございました。              275                                                                               @115770 恐れ入ります。至らない点も多々

In [57]:
reply_counts = (
    conversation_df["customer_tweet_id"]
    .value_counts()
)

print("Customers with multiple Amazon responses:")
print((reply_counts > 1).sum())

print("\nMaximum Amazon responses to one customer tweet:")
print(reply_counts.max())

Customers with multiple Amazon responses:
6308

Maximum Amazon responses to one customer tweet:
6


In [58]:
multi_reply_id = reply_counts[reply_counts > 1].index[0]

print(
    conversation_df[
        conversation_df["customer_tweet_id"] == multi_reply_id
    ][
        [
            "customer_tweet_id",
            "customer_query",
            "amazon_tweet_id",
            "amazon_response"
        ]
    ].to_string(index=False)
)

 customer_tweet_id                                                                                                                    customer_query  amazon_tweet_id                                                                                                                       amazon_response
         1258526.0 @amazonhelp left Delhi on 20th Oct! Expected Delivery date surpassed! Its been 7 days. Where has it gone? https://t.co/DMYcwmwtEJ          1258525 @195271 Please don't provide your order details, we consider it to be personal information. Our page is visible to the public. 3/3^VH
         1258526.0 @amazonhelp left Delhi on 20th Oct! Expected Delivery date surpassed! Its been 7 days. Where has it gone? https://t.co/DMYcwmwtEJ          1258527                @195271 Please report this to our support team here:https://t.co/rS49hgaADF and we will assist you accordingly. 2/3^VH
         1258526.0 @amazonhelp left Delhi on 20th Oct! Expected Delivery date surpassed! Its been 7 

In [59]:
# creating intent classifier dataset 
intent_df = conversation_df[
    [
        "customer_tweet_id",
        "customer_id",
        "customer_query",
        "clean_customer_query",
        "amazon_tweet_id",
        "amazon_response",
        "customer_time"
    ]
].copy()

print("Intent dataset size:", len(intent_df))
print(intent_df.head())

Intent dataset size: 98828
   customer_tweet_id customer_id  \
0              271.0      115770   
1              274.0      115770   
2              616.0      115820   
3              623.0      115824   
4              627.0      115827   

                                      customer_query  \
0  @AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎ...   
1                      @AmazonHelp こちらこそありがとうございました。   
2  @AmazonHelp 3 different people have given 3 di...   
3               @AmazonHelp Okay, danke für die Info   
4  @AmazonHelp @115826 Yeah this is crazy we’re l...   

                                clean_customer_query  amazon_tweet_id  \
0  電話で対応してもらいましたが改良されませんでした。 保証期間も過ぎてるので買い直しになるんで...              273   
1                                  こちらこそありがとうございました。              275   
2  3 different people have given 3 different answ...              618   
3                           Okay, danke für die Info              625   
4  Yeah this is crazy we’re less than a week away...  

In [60]:
intent_df = intent_df[
    intent_df["clean_customer_query"]
    .fillna("")
    .str.strip()
    .ne("")
].copy()

print("After removing empty queries:", len(intent_df))

After removing empty queries: 98828


In [61]:
short_queries = intent_df[
    intent_df["clean_customer_query"].str.len() < 15
]

print("Queries shorter than 15 characters:", len(short_queries))

print(
    short_queries[
        [
            "customer_query",
            "clean_customer_query",
            "amazon_response"
        ]
    ].head(30).to_string(index=False)
)

Queries shorter than 15 characters: 4706
                                  customer_query clean_customer_query                                                                                                                                                                   amazon_response
                         @AmazonHelp All of them          All of them                                                                                   @116611 Hmm.... Let's take a closer look into what's going on here: https://t.co/jzvkhdlrK5 ^ML
                           @AmazonHelp Sadly yes            Sadly yes                                                                                                                          @116617 What devices are you using for Amazon Video? ^AN
                       @AmazonHelp 10 hours ago.        10 hours ago.                                                                                                   @116932 Gotcha- Let us know if you haven't hear

In [62]:
query_counts = (
    intent_df["clean_customer_query"]
    .value_counts()
)

print("Unique customer queries:", query_counts.size)
print("Total queries:", len(intent_df))

print("\nMost common queries:")
print(query_counts.head(20))

Unique customer queries: 89746
Total queries: 98828

Most common queries:
clean_customer_query
Yes            134
Done            83
No              71
AMZL US         66
USPS            61
Thank you       56
Thanks          52
Done.           44
Thank you!      40
Amazon          35
Yes.            32
.com            29
Thanks!         29
UPS             28
Thank you.      25
Merci           24
ありがとうございます！     23
Gracias         23
ありがとうございます      21
Oui             19
Name: count, dtype: int64


In [63]:
non_informative = [
    "yes",
    "yes.",
    "no",
    "no.",
    "done",
    "done.",
    "thanks",
    "thanks!",
    "thanks.",
    "thank you",
    "thank you!",
    "thank you.",
    "merci",
    "gracias",
    "oui",
    "ok",
    "ok.",
    "okay",
    "okay.",
    "amazon",
    ".com"
]
intent_df["query_check"] = (
    intent_df["clean_customer_query"]
    .str.lower()
    .str.strip()
)

intent_df = intent_df[
    ~intent_df["query_check"].isin(non_informative)
].copy()

print("Remaining queries:", len(intent_df))
print("Unique queries:", intent_df["clean_customer_query"].nunique())

Remaining queries: 97997
Unique queries: 89706


In [64]:
print(
    intent_df["clean_customer_query"]
    .value_counts()
    .head(30)
)

print(
    intent_df["clean_customer_query"]
    .sample(100, random_state=42)
    .to_string(index=False)
)

clean_customer_query
AMZL US                                                                                                             66
USPS                                                                                                                61
UPS                                                                                                                 28
ありがとうございます！                                                                                                         23
ありがとうございます                                                                                                          21
Amazon Logistics                                                                                                    19
Amazon.in                                                                                                           18
Amazon logistics                                                                                                    18
Nope                       

In [65]:
intent_df.to_csv("amazon_intent_base.csv", index=False)

print("Saved:", len(intent_df), "rows")

Saved: 97997 rows


In [66]:
import os

print(os.path.exists("amazon_intent_base.csv"))
print(os.path.getsize("amazon_intent_base.csv") / (1024 * 1024), "MB")

True
47.871633529663086 MB


In [67]:
conversation_df.to_csv(
    "amazon_conversation_pairs.csv",
    index=False
)

print("Saved conversation pairs:", len(conversation_df))

Saved conversation pairs: 98828
